# 02. Data Cleaning

이 노트북은 `data/raw/`에 저장된 서울 아파트 매매 실거래가 원본 CSV를 불러와 컬럼 구조를 확인하고, 기본 전처리 및 기본 파생 변수를 생성한다.

## 전처리 범위

- 원본 CSV 로드 (헤더 행 자동 탐지)
- 컬럼 구조와 데이터 크기 확인
- 원본 데이터 수집 기간 및 지역 검증
- 분석에 필요한 컬럼 선별
- 컬럼명 영문 snake_case로 정리
- 거래금액, 전용면적, 층, 건축년도 숫자형 변환
- 거래일자, 거래연도, 거래월 생성
- 구, 법정동 분리 및 검증
- 취소 거래 제외
- `-`로 표시된 빈 문자열을 결측치로 변환
- **이상치 처리** (area_m2, price_10k_krw, age 기준 제거) ← 파생 변수 생성 전에 수행
- 기본 파생 변수 생성: 연식, ㎡당 거래가격, 주소 후보
- 전처리 결과를 `data/interim/`에 저장

## 노트북 역할 범위

| 노트북 | 범위 |
| --- | --- |
| `02_data_cleaning` | 원본 CSV 로드, 기본 컬럼 정리, 결측/이상치 처리, 기본 파생 변수 (연식, ㎡당 가격, 주소 후보) |
| `03_feature_engineering` | 외부 데이터 결합, 거리 기반 변수 생성, 추가 파생 변수 생성 |

## 1. 라이브러리 및 경로 설정


In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DATA_PATH = PROJECT_ROOT / 'data/raw/seoul_apt_trade_2025_raw.csv'
INTERIM_DATA_PATH = PROJECT_ROOT / 'data/interim/seoul_apt_trade_2025_basic_cleaned.csv'

RAW_DATA_PATH.exists()

## 2. 원본 CSV 불러오기

국토교통부 실거래가 공개시스템에서 내려받은 CSV는 상단에 안내 문구와 검색 조건이 포함되어 있다.
헤더 행 위치는 파일 배포 버전마다 달라질 수 있으므로, `skiprows`를 고정값으로 지정하지 않고
`거래금액`·`전용면적` 키워드를 포함한 행을 자동 탐지하여 동적으로 결정한다.
파일 인코딩은 `cp949`로 읽는다.

In [ ]:
def find_header_row(filepath, encoding='cp949', keywords=('거래금액', '전용면적')):
    """실제 컬럼 헤더 행의 인덱스를 자동으로 탐지한다."""
    with open(filepath, encoding=encoding) as f:
        for i, line in enumerate(f):
            if all(kw in line for kw in keywords):
                return i
    raise ValueError(
        f"헤더 행을 찾을 수 없습니다. 파일 형식을 확인하세요. 탐색 키워드: {keywords}"
    )

header_row = find_header_row(RAW_DATA_PATH)
print(f"헤더 행: {header_row + 1}번째 줄에서 탐지 (skiprows={header_row})")

raw_df = pd.read_csv(RAW_DATA_PATH, encoding='cp949', skiprows=header_row)
raw_df.shape

In [ ]:
raw_df.head()

In [ ]:
raw_df.info()

In [ ]:
# 수집 조건 검증: 서울특별시 2025년 데이터만 포함되어야 한다.
years = raw_df['계약년월'].dropna().astype(str).str[:4].unique()
assert (years == '2025').all(), f"2025년 외 연도 포함: {years[years != '2025']}"

assert raw_df['시군구'].dropna().str.startswith('서울특별시').all(), \
    "서울특별시 외 지역 데이터가 포함되어 있습니다."

print(f"✓ 기간 검증 통과: 계약년도 {sorted(years)}")
print(f"✓ 지역 검증 통과: 전체 {len(raw_df):,}건 모두 서울특별시")

## 3. 필요한 컬럼 선별 및 컬럼명 정리


In [ ]:
selected_columns = [
    '시군구',
    '단지명',
    '전용면적(㎡)',
    '계약년월',
    '계약일',
    '거래금액(만원)',
    '층',
    '건축년도',
    '도로명',
]

cancelled_mask = raw_df['해제사유발생일'].notna() & (raw_df['해제사유발생일'].astype(str).str.strip() != '-')
print(f'전체 거래 수: {len(raw_df):,}')
print(f'취소 거래 수: {cancelled_mask.sum():,}')
print(f'취소 거래 제외 후 거래 수: {(~cancelled_mask).sum():,}')

df = raw_df.loc[~cancelled_mask, selected_columns].copy()

df = df.rename(columns={
    '시군구': 'sigungu',
    '단지명': 'apartment_name',
    '전용면적(㎡)': 'area_m2',
    '계약년월': 'contract_ym',
    '계약일': 'contract_day',
    '거래금액(만원)': 'price_10k_krw',
    '층': 'floor',
    '건축년도': 'built_year',
    '도로명': 'road_name',
})

df.head()

## 4. 빈 값 표기 및 기본 타입 변환

원본 데이터에서 `-`는 값이 없음을 나타내는 표기로 사용된다. 분석에 사용할 문자열 컬럼에서는 `-`를 결측치로 변환하고, 숫자형 컬럼은 `pd.to_numeric(..., errors='coerce')`를 통해 변환 불가능한 값을 결측치로 처리한다.


In [ ]:
df['price_10k_krw'] = (
    df['price_10k_krw']
    .astype(str)
    .str.replace(',', '', regex=False)
    .pipe(pd.to_numeric, errors='coerce')
)

string_columns = ['sigungu', 'apartment_name', 'road_name']
for column in string_columns:
    df[column] = df[column].replace('-', pd.NA)

numeric_columns = ['area_m2', 'contract_ym', 'contract_day', 'floor', 'built_year']
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')

df.dtypes

## 5. 지역 및 거래일자 변수 생성


In [ ]:
location_parts = df['sigungu'].str.split(expand=True)
df['sido'] = location_parts[0]
df['gu'] = location_parts[1]
df['law_dong'] = location_parts.iloc[:, 2:].apply(lambda row: ' '.join(row.dropna()), axis=1)

df['contract_year'] = df['contract_ym'] // 100
df['contract_month'] = df['contract_ym'] % 100
df['contract_date'] = pd.to_datetime(
    df['contract_year'].astype('Int64').astype(str) + '-' +
    df['contract_month'].astype('Int64').astype(str).str.zfill(2) + '-' +
    df['contract_day'].astype('Int64').astype(str).str.zfill(2),
    errors='coerce'
)

df[['sigungu', 'sido', 'gu', 'law_dong', 'contract_date']].head()

In [ ]:
# sigungu 분리 결과 검증
assert df['sido'].eq('서울특별시').all(), \
    "sido 컬럼에 '서울특별시' 외 값이 존재합니다."
assert df['gu'].dropna().str.endswith('구').all(), \
    "gu 컬럼 추출 이상: '구'로 끝나지 않는 값이 존재합니다."

print("✓ sigungu 분리 검증 통과")
print(f"  sido 고유값: {df['sido'].unique()}")
print(f"  gu 고유값 수: {df['gu'].nunique()}개")

## 6. 취소 거래 처리 결과 확인

`해제사유발생일`은 최종 분석 변수로 사용하지 않지만, 취소 거래를 제외하기 위해 원본 데이터에서만 임시로 활용했다.


In [ ]:
pd.Series({
    'raw_rows': len(raw_df),
    'cancelled_rows': int(cancelled_mask.sum()),
    'cleaned_rows': len(df),
})

In [ ]:
df.shape

## 7. 이상치 처리

분석에 사용할 수 없는 이상치 행을 제거한다.
`age`는 이 단계에서 먼저 계산하여 음수 여부를 확인한 뒤, `price_per_m2_10k_krw` 생성 전에 제거를 완료한다.

| 조건 | 처리 방침 | 이유 |
| --- | --- | --- |
| `area_m2 <= 0` | 제거 | 유효하지 않은 면적 |
| `price_10k_krw <= 0` | 제거 | 유효하지 않은 거래금액 |
| `contract_date` 결측 | 제거 | 날짜 없이 시계열 분석 불가 |
| `gu` 결측 | 제거 | 지역 변수 없이 분석 불가 |
| `law_dong` 결측 | 제거 | 지역 변수 없이 분석 불가 |
| `age < 0` | 건수 확인 후 제거 | 건축년도 오기재 가능성 높음. 신축·입주예정 케이스 여부 팀 확인 필요 |

In [ ]:
# age 먼저 계산 (이상치 판별에 필요)
df['age'] = df['contract_year'] - df['built_year']

# age < 0: 건수와 내용을 먼저 출력한 뒤 제거
age_anomaly = df[df['age'] < 0]
if len(age_anomaly) > 0:
    print(f"[age < 0] 제거 대상 {len(age_anomaly):,}건 — 원본 건축년도 확인 필요:")
    print(age_anomaly[['apartment_name', 'gu', 'built_year', 'contract_year', 'age']].to_string())
else:
    print("[age < 0] 해당 없음")

before_count = len(df)

df = df[
    (df['area_m2'] > 0) &
    (df['price_10k_krw'] > 0) &
    (df['contract_date'].notna()) &
    (df['gu'].notna()) &
    (df['law_dong'].notna()) &
    (df['age'] >= 0)
].copy()

after_count = len(df)
print(f"\n제거 전: {before_count:,}행 → 제거 후: {after_count:,}행 (제거: {before_count - after_count:,}행)")

## 8. 기본 파생 변수 생성

이상치 제거 완료 후 `price_per_m2_10k_krw`와 `full_road_address`를 생성한다.
`area_m2 <= 0` 행이 제거된 이후이므로 나눗셈 결과에 `inf`나 음수가 발생하지 않는다.

In [ ]:
df['price_per_m2_10k_krw'] = df['price_10k_krw'] / df['area_m2']
df['full_road_address'] = (df['sido'] + ' ' + df['gu'] + ' ' + df['road_name']).where(df['road_name'].notna())

df[[
    'apartment_name', 'gu', 'law_dong', 'contract_date', 'area_m2',
    'floor', 'built_year', 'age', 'price_10k_krw', 'price_per_m2_10k_krw'
]].head()

## 9. 결측치 최종 확인

In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
df[[
    'price_10k_krw', 'price_per_m2_10k_krw', 'area_m2',
    'floor', 'built_year', 'age'
]].describe()

In [ ]:
# 이상치 제거 결과 검증
remaining = df.query('area_m2 <= 0 or price_10k_krw <= 0 or age < 0')
assert len(remaining) == 0, f"이상치 {len(remaining):,}건이 남아있습니다."
print("✓ 이상치 제거 검증 통과")
print(f"최종 데이터: {df.shape[0]:,}행 × {df.shape[1]}열")

## 10. 전처리 결과 저장

기본 전처리 결과는 중간 산출물이므로 `data/interim/`에 저장한다. 해당 폴더의 실제 CSV 파일은 `.gitignore`에 의해 GitHub에 업로드하지 않는다.

In [ ]:
INTERIM_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(INTERIM_DATA_PATH, index=False, encoding='utf-8-sig')
INTERIM_DATA_PATH